# Phase 12a — re-chunking to whole articles, and re-indexing

Phase 2 split long articles at their clauses. `scripts/14. chunk_granularity_evaluation.ipynb` tested that decision against two alternatives and **whole articles won every retrieval metric**, so this notebook rebuilds the corpus and the vector index at article granularity.

| | recall@1 | recall@5 | recall@10 | MRR |
|---|---|---|---|---|
| A — clause chunks (old) | 0.844 | 0.938 | 0.969 | 0.874 |
| **B — whole articles (adopted)** | **0.875** | **0.969** | **1.000** | **0.904** |
| C — clauses merged at retrieval time | 0.844 | 0.938 | 1.000 | 0.879 |

## Why clause-splitting was the wrong unit for this corpus

The original reasoning was that a whole article makes a blurrier embedding, since one vector has to represent several provisions. Reading the actual text disproved that. These articles are **single-topic**: SSL 71 is entirely about military insureds' pensions (29 clauses), SSL 64 entirely about Early Retirement Pension (14), Labor Law 12 entirely about non-Jordanian workers (11). Clauses are sub-rules of one subject, which is simply how statutes are drafted — so merging restores one topic rather than blending many.

Worse, the clause chunks were **semantically incomplete by construction**. Real examples from SSL Article 71:

> *"b. Pension stipulated in Paragraph (a) of this Sub-paragraph shall be increased by (0.5%) for each year of the military insured contributions..."*

> *"a. (50%) of the last monthly wage subject to deduction upon death occurrence."*

Neither can be understood alone — "Paragraph (a)" dangles, and the second never says which benefit it describes. Six clauses of that article say "shall be increased by X%" without naming what is increased, which is exactly why they all looked alike to the retriever and piled into the same result list. On **q22** ("will my pension go up over the years?", ground truth SSL Article 90) the clause index returned Article 71 seven times and never surfaced Article 90 at all; the article index returns Article 90 at **rank 1**.

## Scope of the change

**739 clause chunks → 531 article units.** Only 21 articles are actually split, so 510 units are byte-identical to their old chunk and re-embed to the same vector; the change is targeted, not wholesale.

Rollback is deliberately cheap: the clause-level `chunks.parquet` and the existing `jordan-labor-law` Pinecone index are both left untouched. This notebook writes a **new** parquet and a **new** index, and `src/retrieval.py` is pointed at them — so reverting is a two-line edit.

Known caveat: definitions articles (e.g. SSL Article 2) genuinely are multi-topic glossaries, and merging is not helpful for them. They are barely split to begin with (2 pieces), so B changes little there, but they are the case to watch if retrieval quality on definitional questions regresses.

Requires `COHERE_API_KEY`, `PINECONE_API_KEY` in `.env`. Kernel: `rag311`.

In [1]:
import json
import os
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "retrieval.py").exists()), None)
assert PROJECT, "run this notebook from inside the project tree"
CLAUSE_PATH = PROJECT / "0. cleaned_data" / "chunks.parquet"          # Phase 2 output, left as-is
ARTICLE_PATH = PROJECT / "0. cleaned_data" / "chunks_articles.parquet"  # what production will use

load_dotenv(PROJECT / ".env")

clauses = pd.read_parquet(CLAUSE_PATH)
clauses["article_number"] = clauses["article_number"].astype(str)
clauses["clause"] = clauses["clause"].fillna("").astype(str)
clauses = clauses.reset_index().rename(columns={"index": "doc_order"})

articles = (
    clauses.sort_values("doc_order")
           .groupby(["law", "language", "article_number"], as_index=False)
           .agg(text=("text", lambda s: "\n".join(s)),
                source_file=("source_file", "first"),
                n_clauses=("chunk_id", "size"),
                doc_order=("doc_order", "min"))
           .sort_values("doc_order")
           .reset_index(drop=True)
)
articles["chunk_id"] = articles.law + "_" + articles.language + "_" + articles.article_number
articles["clause"] = ""  # an article unit has no single clause; kept so the schema is unchanged
articles = articles[["chunk_id", "law", "language", "article_number", "clause", "text", "source_file", "n_clauses"]]

# --- sanity checks: nothing lost, nothing duplicated ---
assert articles.chunk_id.is_unique, "duplicate chunk_id after merge"
assert len(articles) == clauses.groupby(["law", "language", "article_number"]).ngroups
text_before = clauses.text.str.len().sum()
text_after = articles.text.str.len().sum() - (articles.n_clauses - 1).sum()  # minus the \n joiners
assert text_before == text_after, f"character count changed: {text_before} -> {text_after}"

print(f"{len(clauses)} clause chunks -> {len(articles)} article units")
print(f"articles that were actually split: {(articles.n_clauses > 1).sum()}")
print(f"unchanged (single-clause) units:   {(articles.n_clauses == 1).sum()}")
print(f"total characters preserved exactly: {text_before:,}")
print()
print(articles.text.str.len().describe(percentiles=[.5, .9, .99]).round(0).to_string())

739 clause chunks -> 531 article units
articles that were actually split: 21
unchanged (single-clause) units:   510
total characters preserved exactly: 477,249

count      531.0
mean       899.0
std       1237.0
min         67.0
50%        534.0
90%       1876.0
99%       6135.0
max      13641.0


In [2]:
# Pinecone caps metadata at 40 KB per vector -- the largest article must fit, since the
# full text is stored as metadata (no separate document store).
sizes = articles.text.map(lambda t: len(t.encode("utf-8")))
print(f"largest article: {sizes.max():,} bytes utf-8 (limit 40,960)")
assert sizes.max() < 38000, "an article is too large for Pinecone metadata"

articles.to_parquet(ARTICLE_PATH, index=False)
print(f"wrote {ARTICLE_PATH.name} ({len(articles)} rows)")
print(articles.nlargest(5, "n_clauses")[["chunk_id", "n_clauses"]].to_string(index=False))

largest article: 17,842 bytes utf-8 (limit 40,960)
wrote chunks_articles.parquet (531 rows)
 chunk_id  n_clauses
ssl_en_71         29
ssl_ar_71         25
ssl_en_11         17
ssl_ar_85         15
ssl_en_85         15


## Embed and index

Same model as Phase 4 (`embed-v4.0`, `input_type="search_document"`, 1536-dim). A **new** index is created so the clause-level one survives for rollback.

In [3]:
import cohere

COHERE_MODEL = "embed-v4.0"
EMBED_DIM = 1536
BATCH = 32          # smaller than Phase 5's 90: some articles are ~14k chars
MAX_RETRIES = 6

co = cohere.ClientV2(api_key=os.environ["COHERE_API_KEY"])

def embed_texts(texts, input_type):
    out = []
    for i in range(0, len(texts), BATCH):
        batch = texts[i:i + BATCH]
        for attempt in range(MAX_RETRIES):
            try:
                r = co.embed(texts=batch, model=COHERE_MODEL, input_type=input_type, embedding_types=["float"])
                out.extend(r.embeddings.float_)
                break
            except Exception as e:
                wait = min(60, 5 * (2 ** attempt))
                print(f"  retry {attempt + 1} at batch {i}: {type(e).__name__} -- waiting {wait}s")
                time.sleep(wait)
        else:
            raise RuntimeError(f"cohere embed failed at batch {i}")
        print(f"  embedded {min(i + BATCH, len(texts))}/{len(texts)}", end="\r")
    return out

vectors = embed_texts(articles.text.tolist(), "search_document")
assert len(vectors) == len(articles) and len(vectors[0]) == EMBED_DIM
print(f"\nembedded {len(vectors)} articles")

  embedded 531/531
embedded 531 articles


In [4]:
from pinecone import Pinecone, ServerlessSpec

INDEX_NAME = "jordan-labor-law-articles"   # new; "jordan-labor-law" (clause-level) is left intact

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
existing = [ix["name"] for ix in pc.list_indexes()]
print("indexes before:", existing)

if INDEX_NAME not in existing:
    pc.create_index(name=INDEX_NAME, dimension=EMBED_DIM, metric="cosine",
                    spec=ServerlessSpec(cloud="aws", region="us-east-1"))
    while not pc.describe_index(INDEX_NAME).status["ready"]:
        time.sleep(1)
    print(f"created index '{INDEX_NAME}'")
else:
    print(f"index '{INDEX_NAME}' already exists -- upserting into it")

index = pc.Index(INDEX_NAME)

payload = [{
    "id": r.chunk_id,
    "values": v,
    "metadata": {"law": r.law, "language": r.language, "article_number": r.article_number,
                 "clause": r.clause, "text": r.text, "source_file": r.source_file,
                 "n_clauses": int(r.n_clauses)},
} for r, v in zip(articles.itertuples(index=False), vectors)]

for i in range(0, len(payload), 100):
    index.upsert(vectors=payload[i:i + 100])
    print(f"  upserted {min(i + 100, len(payload))}/{len(payload)}", end="\r")

time.sleep(3)
print("\n", index.describe_index_stats())

indexes before: ['jordan-labor-law']


created index 'jordan-labor-law-articles'



 DescribeIndexStatsResponse(dimension=1536, total_vector_count=531, metric='cosine', namespaces=1)


## Verify against the live index

q22 is the case that motivated the change: the clause index never surfaced SSL Article 90 for it.

In [5]:
eval_q = {json.loads(l)["question_id"]: json.loads(l)
          for l in open(PROJECT / "eval" / "eval_draft.jsonl", encoding="utf-8")}

for qid in ["q22", "q03", "q33"]:
    q = eval_q[qid]
    emb = embed_texts([q["question_text"]], "search_query")[0]
    res = index.query(vector=emb, top_k=5, include_metadata=True)
    print(f"\n{qid} ({q['question_language']}): {q['question_text'][:80]}")
    print(f"  expected: {q['answer_law']} art {q['answer_article_numbers']} ({q['answer_language']})")
    for m in res["matches"]:
        md = m["metadata"]
        print(f"    [{m['score']:.3f}] {md['law']} art.{md['article_number']} ({md['language']}) "
              f"[{int(md.get('n_clauses', 1))} clause(s) merged]")


q22 (en): Will my pension go up over the years, or does it stay the same amount forever?
  expected: Social_Security_Law art ['90'] (en)
    [0.401] ssl art.90 (en) [1 clause(s) merged]
    [0.388] ssl art.89 (en) [1 clause(s) merged]
    [0.388] ssl art.89 (ar) [1 clause(s) merged]
    [0.382] ssl art.62 (en) [5 clause(s) merged]
    [0.371] ssl art.88 (en) [1 clause(s) merged]



q03 (en): How many paid vacation days do I get per year, and does that go up the longer I 
  expected: Work_Law_1996 art ['61'] (en)
    [0.445] labor96 art.61 (en) [1 clause(s) merged]
    [0.415] labor96 art.61 (ar) [1 clause(s) merged]
    [0.401] labor96 art.66 (en) [1 clause(s) merged]
    [0.352] labor96 art.66 (ar) [1 clause(s) merged]
    [0.350] labor96 art.65 (ar) [1 clause(s) merged]


  embedded 1/1
q33 (en): At what age can I start collecting my retirement pension, and are there other re
  expected: Social_Security_Law art ['62'] (ar)
    [0.440] ssl art.65 (en) [1 clause(s) merged]
    [0.438] ssl art.64 (en) [14 clause(s) merged]
    [0.425] ssl art.62 (en) [5 clause(s) merged]
    [0.406] ssl art.50 (ar) [1 clause(s) merged]
    [0.406] ssl art.62 (ar) [1 clause(s) merged]
